# 10. Community Detection & Summarization Pipeline

Notebook ini menjalankan algoritma **Leiden** (via GDS) untuk mendeteksi komunitas di dalam Knowledge Graph, kemudian men-generate **ringkasan (summary) komunitas** menggunakan LLM.

Ringkasan ini sangat penting untuk mendukung fitur **Global Search** (mendapatkan insight tingkat makro) dan **DRIFT Search**.

In [1]:
import os
import sys

# Tambahkan root direktori ke path agar bisa import dari src
sys.path.append(os.path.dirname(os.getcwd()))

from src.graph.client import GraphClient
from src.config import settings
from src.community.detection import CommunityDetector
from src.community.summarization import CommunitySummarizer
from src.services.providers import get_llm, get_embeddings

import logging
logging.basicConfig(level=logging.INFO)

d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = GraphClient(
    uri=settings.neo4j_uri,
    user=settings.neo4j_user,
    password=settings.neo4j_password
)

INFO:src.graph.client:Connected to Neo4j at bolt://localhost:7687


## Skenario Konfigurasi Graph

Kita menggunakan konfigurasi Pure GraphRAG berdasarkan schema baru.

In [3]:
# ==========================================
# SKENARIO 1: Menggunakan Data Hasil Pure GraphRAG (LLM Extraction)
# ==========================================

NODE_LABELS = [
    'SymptomPattern', 
    'ProblemCluster', 
    'RootCausePattern', 
    'ActionPattern', 
    'Part',
    'MachineModel'
]

RELATIONSHIP_TYPES = [
    'EXHIBITED', 
    'CAUSED_BY', 
    'RESOLVED_BY',
    'INVOLVES_PART',
    'BELONGS_TO'
]

## Step 1: Detect Communities via Leiden
*(Catatan: Membutuhkan plugin Neo4j GDS terinstall)*

In [4]:
detector = CommunityDetector(client)

# Inject label dinamis sesuai skenario yang dipilih di atas
success = detector.detect(
    node_labels=NODE_LABELS,
    relationship_types=RELATIONSHIP_TYPES
)

if success:
    print("✅ Community detection successful!")
else:
    print("❌ Community detection failed. Please check if GDS is installed.")

INFO:src.community.detection:Starting Community Detection using GDS Leiden...
INFO:src.community.detection:Projected graph: 8585 nodes, 18482 relationships.
INFO:src.community.detection:Leiden completed. Found 6982 communities.
INFO:src.community.detection:Modularities across levels: [0.3079350971621286, 0.32915465406427724, 0.33268438278801116]
INFO:src.community.detection:Building Hierarchical Community nodes up to 3 levels...
ERROR:src.community.detection:Failed to run Leiden algorithm: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Unknown function 'toTypeName' (line 3, column 27 (offset: 77))
"        WITH n, CASE WHEN toTypeName(n.communityId) = 'List OF Integer' THEN n.communityId[0] ELSE n.communityId END AS c0_id"
                           ^} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


❌ Community detection failed. Please check if GDS is installed.


## Step 2: Summarize Communities via LLM
Proses ini akan mengirimkan entitas di tiap komunitas ke Ollama untuk dirangkum. Waktu eksekusi bergantung pada jumlah komunitas dan kecepatan Ollama.

In [5]:
llm = get_llm(temperature=0.2)
embedder = get_embeddings()

summarizer = CommunitySummarizer(client, llm, embedder)
summarizer.summarize_all()
print("✅ Community summarization completed!")

INFO:src.community.summarization:Starting Hierarchical Community Summarization...
INFO:src.community.summarization:Summarizing Level 0 communities...
INFO:src.community.summarization:All Level 0 communities already summarized.
INFO:src.community.summarization:Hierarchical Community Summarization completed.


✅ Community summarization completed!


In [6]:
client.close()